# mlops

project

## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = '4z6615tqnogku9'
os.environ['DataZoneDomainId'] = 'dzd-ckmkp9fp6oze0x'
os.environ['DataZoneEnvironmentId'] = '59fi5539m4ud4x'
os.environ['DataZoneDomainRegion'] = 'ap-south-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "4z6615tqnogku9",
                "DataZoneDomainId": "dzd-ckmkp9fp6oze0x",
                "DataZoneEnvironmentId": "59fi5539m4ud4x",
                "DataZoneDomainRegion": "ap-south-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In [0]:
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker import get_execution_role

session = sagemaker.Session()
role = get_execution_role()

estimator = SKLearn(
    entry_point="train.py",
    role=role,
    instance_type="ml.m5.large",
    framework_version="1.0-1"
)

estimator.fit()


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


...

.

.

2026-05-29 10:06:20,426 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-05-29 10:06:20,430 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-05-29 10:06:20,433 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-05-29 10:06:20,456 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-05-29 10:06:20,745 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-05-29 10:06:20,748 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-05-29 10:06:20,766 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-05-29 10:06:20,769 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-05-29 10:06:20,787 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-05-29 10:06:20,79


2026-05-29 10:06:42 Training - Training image download completed. Training in progress.
2026-05-29 10:06:42 Uploading - Uploading generated training model
2026-05-29 10:06:42 Completed - Training job completed


Training seconds: 114
Billable seconds: 114


In [0]:
!python deploy.py

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


...

-

-

-

-

!Model deployed! Endpoint name: sagemaker-scikit-learn-2026-05-29-10-07-26-922


In [0]:
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

endpoint_name = "sagemaker-scikit-learn-2026-05-29-10-07-26-922"

predictor = Predictor(
    endpoint_name=endpoint_name,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer()
)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [0]:
sample = [[17.99, 10.38, 122.8, 1001.0, 0.1184, 0.2776, 0.3001,
           0.1471, 0.2419, 0.07871, 1.095, 0.9053, 8.589, 153.4,
           0.006399, 0.04904, 0.05373, 0.01587, 0.03003, 0.006193,
           25.38, 17.33, 184.6, 2019.0, 0.1622, 0.6656, 0.7119,
           0.2654, 0.4601, 0.1189]]

result = predictor.predict(sample)

print(result)

[0]


In [0]:
predictor.endpoint_name

'sagemaker-scikit-learn-2026-05-29-10-07-26-922'

In [0]:
predictor.delete_endpoint()

## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()